In [1]:
# %pip install pyproj

In [2]:

from pathlib import Path
import re
import pandas as pd
import numpy as np
from pyproj import Transformer


# ==========================================
# 1. Project paths
# ==========================================

PROJECT_DIR = Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
TABLES_DIR = PROJECT_DIR / "output" / "tables"

COORDINATE_FILE = RAW_DIR / "Underground_Stations.csv"
RECOVERY_FILE = TABLES_DIR / "station_recovery_classification.csv"

OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
required_coordinate_columns = [
    "NAME",
    "ATCOCODE",
    "X",
    "Y"
]

In [4]:

# ==========================================
# 2. Load station coordinates
# ==========================================

if not COORDINATE_FILE.exists():
    raise FileNotFoundError(
        f"Coordinate file not found: {COORDINATE_FILE}"
    )

coordinates = pd.read_csv(
    COORDINATE_FILE,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

coordinates.columns = coordinates.columns.str.strip()

coordinates.columns = (
    coordinates.columns
    .str.strip()
)


missing_columns = [
    col for col in required_coordinate_columns
    if col not in coordinates.columns
]

if missing_columns:
    raise ValueError(
        f"Missing coordinate columns: {missing_columns}"
    )

coordinates["X"] = pd.to_numeric(
    coordinates["X"], errors="coerce"
)

coordinates["Y"] = pd.to_numeric(
    coordinates["Y"], errors="coerce"
)

coordinates = coordinates.dropna(
    subset=["X", "Y", "NAME"]
).copy()

In [5]:


# ==========================================
# 3. Convert British National Grid to WGS84
# ==========================================

transformer = Transformer.from_crs(
    "EPSG:27700",
    "EPSG:4326",
    always_xy=True
)

coordinates["Longitude"], coordinates["Latitude"] = (
    transformer.transform(
        coordinates["X"].to_numpy(),
        coordinates["Y"].to_numpy()
    )
)

print("Coordinate records:", len(coordinates))
print(
    coordinates[
        ["NAME", "Longitude", "Latitude"]
    ].head()
)


Coordinate records: 273
            NAME  Longitude   Latitude
0     St. Paul's  -0.097518  51.514953
1       Mile End  -0.033768  51.525235
2  Bethnal Green  -0.054690  51.527238
3         Leyton  -0.005487  51.556604
4    Snaresbrook   0.021393  51.580677


In [6]:
# ==========================================
# Handle Bank and Monument combined station
# ==========================================

bank_monument_xy = (
    coordinates.loc[
        coordinates["NAME"].isin(["Bank", "Monument"]),
        ["X", "Y"]
    ].mean()
)

bank_monument_row = {
    "NAME": "Bank and Monument",
    "ATCOCODE": "Bank_Monument",
    "X": bank_monument_xy["X"],
    "Y": bank_monument_xy["Y"],
}

coordinates = pd.concat(
    [
        coordinates,
        pd.DataFrame([bank_monument_row])
    ],
    ignore_index=True
)

In [7]:


# ==========================================
# 4. Standardise station names
# ==========================================

def normalise_station_name(name):
    """Create a consistent key for station-name matching."""

    name = str(name).lower().strip()

    # Remove punctuation
    name = re.sub(r"[^\w\s]", "", name)

    # Remove common suffix
    name = re.sub(r"\bstation\b", "", name)

    # Standardise whitespace
    name = re.sub(r"\s+", " ", name).strip()

    return name


coordinates["Station_Key"] = (
    coordinates["NAME"].apply(normalise_station_name)
)



In [8]:
# Explicit station name crosswalk
station_name_mapping = {
    "Paddington TfL": "Paddington",
    "Tottenham Hale LU": "Tottenham Hale",
    "Hammersmith (H&C)": "Hammersmith (H&C Line)",
    "Waterloo LU": "Waterloo",
    "Liverpool Street LU": "Liverpool Street",
    "Heathrow Terminal 5 LU": "Heathrow Terminal 5",
    "Edgware Road (DIS)": "Edgware Road (Circle Line)",
    "Heathrow Terminal 4 LU": "Heathrow Terminal 4",
    "Euston LU": "Euston",
    "West Hampstead LU": "West Hampstead",
    "Hammersmith (DIS)": "Hammersmith (Dist&Picc Line)",
    "Balham LU": "Balham",
    "Charing Cross LU": "Charing Cross",
    "London Bridge LU": "London Bridge",
    "Heathrow Terminals 123 LU": "Heathrow Terminals 1-2-3",
    "Victoria LU": "Victoria",
    "Edgware Road (Bak)": "Edgware Road (Bakerloo)",
    "Blackfriars LU": "Blackfriars",
    "Canary Wharf LU": "Canary Wharf",
    "Marylebone LU": "Marylebone",
    "Vauxhall LU": "Vauxhall",
    "Cannon Street LU": "Cannon Street",
    "Elephant & Castle LU": "Elephant & Castle",
    "Shepherd's Bush LU": "Shepherd's Bush",
    "Bethnal Green LU": "Bethnal Green",
    "Brixton LU": "Brixton",
}

In [9]:
# ==========================================
# 5. Load Part 01 station recovery data
# ==========================================

if not RECOVERY_FILE.exists():
    raise FileNotFoundError(
        f"Recovery file not found: {RECOVERY_FILE}"
    )

recovery = pd.read_csv(RECOVERY_FILE)

# Rename columns to standard format
recovery = recovery.rename(columns={
    "2019": "Flow_2019",
    "2022": "Flow_2022",
    "2025": "Flow_2025",
    "Recovery Index": "Recovery_Index",
    "Recovery Group": "Recovery_Group"
})

required_recovery_columns = [
    "Station",
    "Flow_2019",
    "Flow_2022",
    "Flow_2025",
    "Recovery_Index"
]

missing_columns = [
    col for col in required_recovery_columns
    if col not in recovery.columns
]

if missing_columns:
    raise ValueError(
        f"Missing recovery columns: {missing_columns}\n"
        f"Available columns: {recovery.columns.tolist()}"
    )

# Convert numeric columns
numeric_columns = [
    "Flow_2019",
    "Flow_2022",
    "Flow_2025",
    "Recovery_Index"
]

for col in numeric_columns:
    recovery[col] = pd.to_numeric(
        recovery[col], errors="coerce"
    )

# Standardise station names
recovery["Station_Key"] = (
    recovery["Station"].apply(normalise_station_name)
)

print("Recovery records:", len(recovery))
print("\nColumns:")
print(recovery.columns.tolist())

print("\nPreview:")
print(
    recovery[
        ["Station", "Flow_2019", "Flow_2022",
         "Flow_2025", "Recovery_Index"]
    ].head()
)

Recovery records: 267

Columns:
['Station_ID', 'Flow_2019', 'Flow_2022', 'Flow_2025', 'Station', 'Change (%)', 'Recovery_Index', 'Recovery_Group', 'Station_Key']

Preview:
                Station  Flow_2019      Flow_2022      Flow_2025  \
0            Farringdon    98706.0  119435.975000  163498.556818   
1              Richmond    24618.0   11662.884493   39606.955080   
2  Tottenham Court Road   132723.0  158194.225000  191246.419643   
3             Wimbledon    40525.0   38176.908165   56818.920905   
4           Whitechapel    44865.0   49736.291667   61081.208333   

   Recovery_Index  
0      165.641964  
1      160.886161  
2      144.094407  
3      140.207084  
4      136.144452  


In [10]:
# ==========================================
# 6. Consolidate duplicate station coordinates
# ==========================================

# Group records belonging to the same ATCOCODE
coordinates = (
    coordinates
    .groupby("ATCOCODE", as_index=False)
    .agg({
        "X": "mean",
        "Y": "mean",
        "NAME": "first",
        "LINES": "first",
        "ACCESSIBILITY": "first"
    })
)

# Recalculate longitude and latitude using the averaged coordinates
coordinates["Longitude"], coordinates["Latitude"] = (
    transformer.transform(
        coordinates["X"].to_numpy(),
        coordinates["Y"].to_numpy()
    )
)

# Rebuild station matching keys
coordinates["Station_Key"] = (
    coordinates["NAME"].apply(normalise_station_name)
)

print("Coordinate records after consolidation:", len(coordinates))

Coordinate records after consolidation: 271


In [11]:

# ==========================================
# 6. Check duplicate station names
# ==========================================

if coordinates["Station_Key"].duplicated().any():
    duplicates = coordinates.loc[
        coordinates["Station_Key"].duplicated(keep=False),
        ["NAME", "ATCOCODE", "Station_Key"]
    ]

    raise ValueError(
        "Duplicate station names in coordinate data. "
        "Review these records before merging:\n"
        f"{duplicates.to_string(index=False)}"
    )

if recovery["Station_Key"].duplicated().any():
    duplicates = recovery.loc[
        recovery["Station_Key"].duplicated(keep=False),
        ["Station", "Station_Key"]
    ]

    raise ValueError(
        "Duplicate station names in recovery data. "
        "Review these records before merging:\n"
        f"{duplicates.to_string(index=False)}"
    )



In [12]:
recovery["Station_Match"] = (
    recovery["Station"]
    .map(station_name_mapping)
    .fillna(recovery["Station"])
)

recovery["Station_Key"] = recovery["Station_Match"].apply(
    normalise_station_name
)

coordinates["Station_Key"] = coordinates["NAME"].apply(
    normalise_station_name
)

In [13]:

# ==========================================
# 7. Merge recovery data with coordinates
# ==========================================

map_df = recovery.merge(
    coordinates[
        [
            "Station_Key",
            "NAME",
            "X",
            "Y",
            "Longitude",
            "Latitude",
            "LINES",
            "ATCOCODE",
            "ACCESSIBILITY"
        ]
    ],
    on="Station_Key",
    how="left",
    indicator=True
)

matched = recovery.merge(
    coordinates,
    on="Station_Key",
    how="left",
    indicator=True
)

unmatched = map_df[
    map_df["_merge"] == "left_only"
].copy()

matched = matched.drop(columns=["_merge"])

print("\nMatching results")
print("Recovery stations:", len(recovery))
print("Matched stations:", len(matched))
print("Unmatched stations:", len(unmatched))



Matching results
Recovery stations: 267
Matched stations: 267
Unmatched stations: 0


In [14]:
# ==========================================
# 8. Recovery classification
# ==========================================

matched["Recovery_Group"] = matched["Recovery_Group"].fillna(
    pd.cut(
        matched["Recovery_Index"],
        bins=[-float("inf"), 90, 100, float("inf")],
        labels=[
            "Below baseline",
            "Near baseline",
            "Above baseline"
        ],
        right=False
    )
)

print("Recovery group counts:")
print(matched["Recovery_Group"].value_counts())

Recovery group counts:
Recovery_Group
Below baseline    223
Near baseline      23
Above baseline     21
Name: count, dtype: int64


In [15]:

# ==========================================
# 9. Export results
# ==========================================

matched.to_csv(
    OUTPUT_DIR / "part3_map_ready.csv",
    index=False
)

unmatched.to_csv(
    OUTPUT_DIR / "part3_unmatched_stations.csv",
    index=False
)

coordinates.to_csv(
    OUTPUT_DIR / "part3_coordinates_clean.csv",
    index=False
)


In [16]:


# ==========================================
# 10. Final audit
# ==========================================

print("\nRecovery group counts:")
print(
    matched["Recovery_Group"].value_counts(
        dropna=False
    )
)

print("\nMissing coordinate values:")
print(
    matched[
        ["Longitude", "Latitude"]
    ].isna().sum()
)

print("\nPart 03 preprocessing complete.")
print("Output directory:", OUTPUT_DIR)



Recovery group counts:
Recovery_Group
Below baseline    223
Near baseline      23
Above baseline     21
Name: count, dtype: int64

Missing coordinate values:
Longitude    0
Latitude     0
dtype: int64

Part 03 preprocessing complete.
Output directory: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/data/processed
